# Nested Learning for CAD — CADIC gốc/tối ưu, đối chiếu chi phí T4

Đọc `docs/main_spec/cadic_cost_validation.md`. Không thay method/protocol, không anomaly test hay VisA, không tiếp tục benchmark. CADIC dựng predecessor riêng từ179 normals rồi R1/O1/O2/R2 trên16 ảnh179–194; PREP báo riêng. So historical NESTED là cross-session descriptive, bank trajectories khác; không là whole-stream ETA hay accuracy gain.

**4 inputs:** source `hope_cad_cadic_cost_source_r1.zip`; original reviewed NESTED_seed0 paused376 evidence; MVTec AD raw; locked ViT-B/8 safetensors. Tháo source/output profiler, NN-validation và bridge khác. Không cần attach G3 output, các số tham chiếu đã pin SHA trong source.

**GPU T4×2, kernel mới, Save Version → Save & Run All.** `RESUME_OVERRIDE=None`; soft180min gồm180s export reserve, không ETA. Không tự retry khi partial/fail. Runtime phải khớp canonical T4; không pip upgrade. Một GPU cuda:0 được sử dụng. Heartbeat subprocess30s; progress mỗi8ảnh. Chi phí179ảnh chuẩn bị có thể đáng kể.

Lưu ZIP `hope_cad_cadic_cost_evidence.zip` và log tại `D:/src/DATN/.agents/artifacts/runs/support_v5_1_cadic_cost_01/session_01/`. Complete cần review source/runtime/parity/timing. Không thay output bridge hoặc resume benchmark bằng clone này.


In [ ]:
PHASE='cadic_cost'
SOURCE_OVERRIDE=None
PARENT_OVERRIDE=None  # exact reviewed benchmark376 ZIP or extracted evidence root
RESUME_OVERRIDE=None  # unsupported: preserve failures for review; keep None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

from pathlib import Path,PurePosixPath
import hashlib,json,zipfile,time,os,sys,shutil
STARTED=time.monotonic()
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
sys.dont_write_bytecode=True
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_cadic_cost_source_r1')
WORK=Path('/kaggle/working/hope_cadic_cost')
ASSETS=Path('/tmp/hope_throughput_assets')
OUTPUT=Path('/kaggle/working/hope_cad_cadic_cost_evidence.zip')
PINNED_CONTRACT='a06143f04a353a8e01ec295a2103c70aaa0886b0dce059381254a0c81f59793e'
print('[CADIC COST BOOTSTRAP] locating source; elapsed',round(time.monotonic()-STARTED,1),'s',flush=True)
def safe(root,name):
    p=PurePosixPath(name)
    if p.is_absolute() or p.as_posix()!=name or '..' in p.parts or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    out=root/name
    if not out.resolve().is_relative_to(root.resolve()):raise ValueError('Escaping source path')
    return out
def manifest(p):
    if p.is_dir():return json.loads((p/'cadic_cost_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:return json.loads(z.read('cadic_cost_manifest.json'))
choices=[Path(SOURCE_OVERRIDE)] if SOURCE_OVERRIDE else [p.parent for p in INPUT.rglob('cadic_cost_manifest.json')]+list(INPUT.rglob('*.zip'))
valid=[]
for p in choices:
    try:
        m=manifest(p)
        if m.get('schema')=='hope_cadic_cost_source_v1':valid.append((p,m))
    except (OSError,KeyError,ValueError,zipfile.BadZipFile):
        if SOURCE_OVERRIDE:raise
identities={json.dumps(m,sort_keys=True) for p,m in valid}
if len(identities)!=1:raise ValueError('Expected ONE CADIC cost source identity; set SOURCE_OVERRIDE')
source,m=sorted(valid,key=lambda v:str(v[0]))[0]
if m['contract_sha256']!=PINNED_CONTRACT:raise ValueError('Notebook/source contract version mismatch')
ROOT.mkdir(parents=True,exist_ok=True)
z=None if source.is_dir() else zipfile.ZipFile(source)
try:
    if z and (len(z.namelist())!=len(set(z.namelist())) or set(z.namelist())!=set(m['files'])|{'cadic_cost_manifest.json'}):raise ValueError('Source ZIP inventory mismatch')
    for name,expected in m['files'].items():
        original=safe(source,name) if source.is_dir() else None
        if original is not None and original.is_symlink():raise ValueError('Source symlink')
        raw=original.read_bytes() if original is not None else z.read(name)
        if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Source payload SHA mismatch: '+name)
        dest=safe(ROOT,name);dest.parent.mkdir(parents=True,exist_ok=True)
        if dest.exists() and dest.read_bytes()!=raw:raise ValueError('Existing source differs; use a fresh kernel/session')
        dest.write_bytes(raw)
    target=ROOT/'cadic_cost_manifest.json';raw=json.dumps(m,sort_keys=True).encode()
    if target.exists() and json.loads(target.read_bytes())!=m:raise ValueError('Existing delivery differs')
    target.write_bytes(raw)
finally:
    if z:z.close()
for name in ('training','models','dataset','notebook_support'):
    module=sys.modules.get(name)
    if module:
        paths=list(getattr(module,'__path__',[]))
        if not paths or any(not Path(p).resolve().is_relative_to(ROOT) for p in paths):raise ValueError('Mixed source imports; restart kernel')
sys.path.insert(0,str(ROOT));os.chdir(ROOT)
print('[CADIC COST BOOTSTRAP] verified',len(m['files']),'files;',m['origin'],flush=True)
print('Soft180min including export reserve; PREP179 separately then CADIC R1/O1/O2/R2 on16 known normals; no anomaly evaluation or adoption',flush=True)


In [ ]:
from notebook_support.cadic_cost_validation import run_notebook
RESULT=run_notebook(dict(inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),started=STARTED,
    PARENT_OVERRIDE=PARENT_OVERRIDE,RESUME_OVERRIDE=RESUME_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
